<a href="https://colab.research.google.com/github/evie1706/EEG/blob/main/Colab_01_Foundations_and_Primer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 📘 Pre-Workshop Notebook 1: Foundations of Brain and Behavioral Research Workflows
## Scientific Workflows for Brain and Behavioral Research
### National Science Foundation Supported Learning Initiative (Award No. OAC-2417875)

---

## 📋 Set Up Your Profile
*Run the cell below to record your name and institution in this notebook.*

In [1]:
# PROFILE SETUP
student_name = "Dr. Evie A. Malaia"
institution = "University of Alabama"
research_area = "Neuroscience/Speech-Language Pathology"

print(f"✅ Notebook initialized for {student_name} ({institution})")
print(f"Research area: {research_area}")

✅ Notebook initialized for Dr. Evie A. Malaia (University of Alabama)
Research area: Neuroscience/Speech-Language Pathology


---

## 🤖 Using AI Tools to Build Your Workflow ("Ask → Build → Document")

Welcome to a different way of learning to work with data! In this notebook, you will not be memorizing programming syntax. Instead, you will act as a **Workflow Designer** — someone who decides *what* a workflow should do and then uses AI tools (such as Claude, ChatGPT, or GitHub Copilot) to help generate the code.

To get the most out of each activity, follow this three-step cycle:
1. **Ask:** Copy the suggested prompt below into your preferred AI tool. Adjust it to fit your research area if you like.
2. **Build:** Paste the AI-generated code into the empty code cell below the prompt, then run it and look at what it produces.
3. **Document:** In the reflection box that follows, write in plain language what the code did and what it taught you about the concept.

---
## 🧠 Part A: How Brain Data Is Captured and Used
### Topic: Brain-Machine Interfaces (BMIs), Brain Imaging (fMRI), and Real-Time Systems

This section introduces two of the most important types of brain and behavioral data used in modern neuroscience research, and how analysis workflows process them.

### 🧭 Activity A.1 — Simulating a Brain-Machine Interface Decoder

> *Copy the prompt below into your preferred AI tool, then paste the generated code into the code cell beneath it.*
>
> "Act as a neuroscience research assistant helping a biology student understand brain-machine interfaces. Write a Python script using NumPy and Matplotlib that simulates how brain signals from the motor cortex can be used to control a robotic limb. The script should:
> 1. Generate 2 seconds of synthetic data representing voltage recordings from 64 brain electrodes, sampled 1000 times per second, including realistic background noise and a few sudden signal spikes representing muscle movements.
> 2. Write a simple decoding function that converts these raw voltage patterns into X and Y position coordinates representing the intended movement of a robotic limb.
> 3. Plot the raw electrode signals and the resulting decoded movement path side-by-side.
> Add plain-language comments throughout explaining what each step does and what the numbers represent."

In [5]:
import numpy as np
import matplotlib.pyplot as plt

# ------------------------------------------------------------
# Brain-machine interface demo
# ------------------------------------------------------------
# We simulate:
#   - 64 electrodes recording activity from motor cortex
#   - 2 seconds of recording
#   - 1,000 samples per second (1 kHz)
#   - noisy background brain activity
#   - several brief bursts ("spikes") associated with movement
#
# This is a simplified teaching example, NOT a realistic neural
# decoding algorithm used in a clinical brain-machine interface.
# ------------------------------------------------------------

# Reproducible random numbers, so the same simulation is produced
# each time the script is run.
rng = np.random.default_rng(42)

# Basic recording parameters
n_electrodes = 64       # Number of electrodes
sample_rate = 1000      # Samples per second (1 kHz)
duration = 2.0          # Recording duration in seconds
n_samples = int(sample_rate * duration)

# Time of every voltage measurement
time = np.arange(n_samples) / sample_rate

# ------------------------------------------------------------
# 1. Create background voltage recordings
# ------------------------------------------------------------

# Start with random background noise.
# "0.5" represents the typical size of the simulated voltage
# fluctuations. Real neural recordings have much more complicated
# noise and signal characteristics.
signals = rng.normal(
    loc=0.0,
    scale=0.5,
    size=(n_electrodes, n_samples)
)

# Add a slow oscillation to each electrode. This makes the
# simulated recordings look a little more like changing biological
# signals rather than completely independent random noise.
for electrode in range(n_electrodes):
    frequency = rng.uniform(5, 15)  # Hz
    phase = rng.uniform(0, 2 * np.pi)

    signals[electrode] += (
        0.15
        * np.sin(2 * np.pi * frequency * time + phase)
    )

# ------------------------------------------------------------
# 2. Add sudden bursts of neural activity
# ------------------------------------------------------------

# These are artificial "movement-related" events.
# Each event has:
#   time     = when the movement-related burst occurs
#   strength = how strong the burst is
#
# In a real experiment, researchers would record action potentials
# and/or local field potentials rather than inserting artificial
# spikes like this.
movement_events = [
    (0.35, 3.0),
    (0.85, 2.5),
    (1.25, 4.0),
    (1.70, 3.2),
]

for event_time, strength in movement_events:
    center = int(event_time * sample_rate)

    # Make each event last about 20 milliseconds.
    width = int(0.020 * sample_rate)

    start = max(0, center - width // 2)
    end = min(n_samples, center + width // 2)

    # A bell-shaped burst is more visually useful than one isolated
    # sample. It represents a short period of increased neural activity.
    burst_time = time[start:end] - event_time
    burst = strength * np.exp(
        -(burst_time ** 2) / (2 * 0.006 ** 2)
    )

    # Only some electrodes respond strongly to each movement.
    # This creates different spatial patterns across the electrode array.
    active_electrodes = rng.choice(
        n_electrodes,
        size=20,
        replace=False
    )

    for electrode in active_electrodes:
        # Each electrode gets a slightly different response.
        gain = rng.uniform(0.5, 1.2)
        signals[electrode, start:end] += gain * burst


# ------------------------------------------------------------
# 3. Simple decoding function
# ------------------------------------------------------------

def decode_movement(voltage_data, sample_rate):
    """
    Convert the 64 electrode recordings into X/Y movement.

    This is intentionally simple:
      1. Calculate the amount of activity across electrodes.
      2. Divide electrodes into two groups.
      3. Use their activity to estimate X and Y movement.

    In a real brain-machine interface, a decoder might use
    regression, Kalman filters, neural networks, or other
    statistical models trained on recorded movement data.
    """

    # Take the absolute voltage because both positive and negative
    # fluctuations can represent increased activity in this toy model.
    activity = np.abs(voltage_data)

    # Average activity over short 20-ms windows.
    window = int(0.020 * sample_rate)

    n_windows = voltage_data.shape[1] // window

    binned_activity = activity[:, :n_windows * window].reshape(
        voltage_data.shape[0],
        n_windows,
        window
    ).mean(axis=2)

    # First half of the electrodes contributes mainly to X.
    # Second half contributes mainly to Y.
    x_signal = binned_activity[:32].mean(axis=0)
    y_signal = binned_activity[32:].mean(axis=0)

    # Remove the baseline level so that only changes in activity
    # contribute strongly to the movement.
    x_signal -= np.mean(x_signal)
    y_signal -= np.mean(y_signal)

    # Scale the signals to create a convenient movement size.
    x_velocity = x_signal * 0.025
    y_velocity = y_signal * 0.025

    # Integrate velocity over time to obtain position.
    # This means that stronger neural activity produces faster
    # movement of the simulated robotic limb.
    dt = window / sample_rate

    x_position = np.cumsum(x_velocity) * dt
    y_position = np.cumsum(y_velocity) * dt

    # Normalize the resulting path so it fits nicely on the plot.
    x_position /= max(np.max(np.abs(x_position)), 1e-9)
    y_position /= max(np.max(np.abs(y_position)), 1e-9)

    return x_position, y_position


# Run the decoder
x_position, y_position = decode_movement(
    signals,
    sample_rate
)

# Time associated with each decoded position
decoder_window = 0.020
decoded_time = np.arange(len(x_position)) * decoder_window


# ------------------------------------------------------------
# 4. Plot the simulated neural recordings and movement
# ------------------------------------------------------------

fig, axes = plt.subplots(
    1, 2,
    figsize=(14, 6)
)

# ---- Left: electrode recordings ----

ax = axes[0]

# Plot all 64 electrodes with small vertical offsets so that
# individual recordings can be seen.
offset = 3.0

for electrode in range(n_electrodes):
    ax.plot(
        time,
        signals[electrode] + electrode * offset,
        linewidth=0.5,
        alpha=0.7
    )

ax.set_title("Simulated Motor-Cortex Electrode Signals")
ax.set_xlabel("Time (seconds)")
ax.set_ylabel("Electrode number / voltage (offset)")
ax.grid(alpha=0.2)

# ---- Right: decoded robotic-limb path ----

ax = axes[1]

ax.plot(
    x_position,
    y_position,
    color="tab:blue",
    linewidth=2,
    label="Decoded limb path"
)

# Mark the starting point.
ax.scatter(
    x_position[0],
    y_position[0],
    color="green",
    s=80,
    label="Start",
    zorder=3
)

# Mark the ending point.
ax.scatter(
    x_position[-1],
    y_position[-1],
    color="red",
    s=80,
    label="End",
    zorder=3
)

ax.set_title("Decoded Robotic-Limb Movement")
ax.set_xlabel("X position (normalized units)")
ax.set_ylabel("Y position (normalized units)")
ax.set_aspect("equal", adjustable="box")
ax.grid(alpha=0.3)
ax.legend()

plt.suptitle(
    "Simplified Brain-Machine Interface Simulation",
    fontsize=14
)

plt.tight_layout()
plt.show()
plt.savefig("bmi_simulation.png", dpi=300, bbox_inches="tight")
plt.show()



In [4]:
# 💻 Activity A.1 — Paste your AI-generated BMI Decoder code here:
"""
=============================================================================
 BRAIN-MACHINE INTERFACE (BMI) SIMULATION
 A toy model of how motor cortex activity can steer a robotic limb.
=============================================================================

THE BIG PICTURE
---------------
When you reach for a coffee cup, neurons in your primary motor cortex (M1)
fire. Each neuron has a "preferred direction": it fires fastest when the arm
moves one particular way, and less as the movement angle drifts away from it.
No single neuron knows where the arm is going, but the *population* does.

A BMI exploits this. Stick an electrode array in M1, record the voltages,
measure how hard each electrode's neurons are firing, and average their
preferred directions together, weighted by firing rate. The result is a
2D vector pointing where the person intends to move. Feed that to a robot arm.

This script fakes the whole pipeline end to end:
    (1) generate realistic-looking electrode voltages
    (2) decode them into X/Y movement
    (3) plot both sides of the story

Nothing here touches real neural data — it's a sandbox for building intuition.
=============================================================================
"""

import numpy as np
import matplotlib
matplotlib.use("Agg")  # render to file instead of a pop-up window
import matplotlib.pyplot as plt
from matplotlib.collections import LineCollection

# A seeded random generator: same "random" numbers every run, so your plots
# are reproducible. Change 42 to any integer to get a different recording.
rng = np.random.default_rng(42)


# =============================================================================
# SECTION 0 — RECORDING SETTINGS
# =============================================================================

FS = 1000                       # sampling rate: 1000 measurements per second
DURATION = 2.0                  # seconds of recording
N_SAMPLES = int(FS * DURATION)  # 2000 total time points per electrode
N_CHANNELS = 64                 # 64 electrodes, i.e. an 8x8 implanted grid
DT = 1.0 / FS                   # 0.001 s = the time gap between two samples

# Time axis in seconds: [0.000, 0.001, 0.002, ..., 1.999]
time = np.arange(N_SAMPLES) * DT

# Each electrode gets a preferred direction, spread evenly around the compass
# (0 to 2*pi radians) with a little random jitter so it isn't artificially neat.
# This is the "tuning" that makes decoding possible at all.
preferred_dirs = (
    np.linspace(0, 2 * np.pi, N_CHANNELS, endpoint=False)
    + rng.normal(0, 0.25, N_CHANNELS)
)

# Movement events the "subject" attempts during these 2 seconds.
# Each one is (start time in s, duration in s, direction in radians).
# 0 rad = right, pi/2 = up, pi = left, 3*pi/2 = down.
MOVEMENTS = [
    (0.30, 0.25, np.deg2rad(20)),    # reach right and slightly up
    (0.85, 0.25, np.deg2rad(115)),   # reach up and to the left
    (1.40, 0.25, np.deg2rad(300)),   # reach down and to the right
]


# =============================================================================
# SECTION 1 — GENERATE THE SYNTHETIC RECORDING
# =============================================================================

def spike_waveform(fs=FS):
    """
    The stereotyped shape of a single neuron firing, ~2 ms long.

    A real extracellular action potential looks like a sharp negative dip
    followed by a slower positive rebound. We approximate that with the
    negative first derivative of a Gaussian. Amplitude is in microvolts (uV).
    """
    n = int(0.002 * fs)                    # 2 ms => 2 samples at 1 kHz
    n = max(n, 4)                          # keep at least 4 points to draw a shape
    x = np.linspace(-2.5, 2.5, n)
    wave = -x * np.exp(-(x ** 2) / 2)      # dip-then-rebound
    return 90.0 * wave / np.max(np.abs(wave))   # scale peak to ~90 uV


def firing_rate_profile(channel_idx):
    """
    How fast this electrode's neurons fire over time, in spikes per second.

    Two ingredients:
      - a baseline "idle chatter" rate that never stops
      - a bump during each movement, scaled by how well the movement direction
        matches this electrode's preferred direction (cosine tuning)

    Returns an array of length N_SAMPLES.
    """
    baseline = 8.0                              # ~8 spikes/s when at rest
    rate = np.full(N_SAMPLES, baseline)

    for start, dur, direction in MOVEMENTS:
        # Cosine tuning: 1.0 if the movement is exactly this electrode's
        # favourite direction, 0.0 if it's 90 degrees off, and we clip
        # negatives to 0 (a neuron can't fire a negative number of spikes).
        alignment = np.cos(direction - preferred_dirs[channel_idx])
        alignment = max(alignment, 0.0)

        # A smooth bell-shaped burst centred on the movement, not a square
        # block — real neural activity ramps up and back down.
        centre = start + dur / 2
        width = dur / 2.2
        envelope = np.exp(-0.5 * ((time - centre) / width) ** 2)

        rate += 120.0 * alignment * envelope     # up to ~120 extra spikes/s

    return rate


def generate_recording():
    """
    Build the raw voltage matrix.

    Returns a (64, 2000) array: one row per electrode, one column per
    millisecond. Values are in microvolts, the same units a real amplifier
    would report.
    """
    signals = np.zeros((N_CHANNELS, N_SAMPLES))
    wave = spike_waveform()

    for ch in range(N_CHANNELS):
        # --- Ingredient A: spikes ------------------------------------------
        # Convert a firing rate into actual discrete spike times. In each 1 ms
        # bin, the chance of a spike is rate * DT. Comparing that probability
        # against a uniform random number gives a Poisson-like spike train.
        rate = firing_rate_profile(ch)
        spike_train = (rng.random(N_SAMPLES) < rate * DT).astype(float)

        # Stamp the spike waveform down at every spike time. Convolution is
        # just "paste this shape wherever there's a 1", done efficiently.
        spikes = np.convolve(spike_train, wave, mode="same")

        # --- Ingredient B: background noise --------------------------------
        # Thermal/electronic noise from the amplifier and distant neurons.
        # ~15 uV of white noise is typical for a chronic cortical array.
        white = rng.normal(0, 15.0, N_SAMPLES)

        # Slow drift: the electrode-tissue interface wanders over hundreds of
        # milliseconds. Modelled as a low-frequency sine with a random phase.
        drift = 20.0 * np.sin(2 * np.pi * 1.5 * time + rng.uniform(0, 2 * np.pi))

        # Mains hum: 60 Hz interference leaking in from the building's wiring.
        # A constant annoyance in every real electrophysiology rig.
        hum = 6.0 * np.sin(2 * np.pi * 60 * time + rng.uniform(0, 2 * np.pi))

        signals[ch] = spikes + white + drift + hum

    return signals


# =============================================================================
# SECTION 2 — THE DECODER
# =============================================================================
#
# Turning voltages into intent happens in four steps:
#   1. Strip out slow drift so only fast spiking energy remains.
#   2. Measure how much spiking energy each electrode has, moment to moment.
#   3. Convert that into a "how excited is this electrode vs. its own resting
#      state" score, so a loud electrode doesn't dominate a quiet one.
#   4. Add up all 64 preferred-direction arrows, weighted by those scores.
#      That sum is the intended velocity. Integrate velocity to get position.
# =============================================================================

def high_pass(signals, cutoff_hz=100.0):
    """
    Remove slow wandering (drift, low-frequency brain rhythms) and keep the
    fast stuff where spikes live.

    Trick used here: smooth the signal heavily to estimate the slow part,
    then subtract it. Smoothing window length is set by the cutoff frequency.
    This is a crude but dependency-free stand-in for a proper Butterworth
    filter (scipy.signal.butter would be the real-world choice).
    """
    win_len = max(int(FS / cutoff_hz), 3)          # 100 Hz -> 10-sample window
    kernel = np.ones(win_len) / win_len

    slow = np.array([np.convolve(row, kernel, mode="same") for row in signals])
    return signals - slow


def spiking_power(signals, window_ms=50):
    """
    Estimate spiking energy per electrode over time.

    Rectify (square the voltage so dips and peaks both count as activity),
    then average within a sliding window. A 50 ms window is the usual
    compromise: long enough to smooth out randomness, short enough that the
    robot arm still feels responsive.

    Returns a (64, 2000) array of power values.
    """
    win_len = int(FS * window_ms / 1000.0)         # 50 ms -> 50 samples
    kernel = np.ones(win_len) / win_len

    rectified = signals ** 2
    return np.array([np.convolve(row, kernel, mode="same") for row in rectified])


def normalise_to_baseline(power, baseline_end_s=0.25):
    """
    Convert raw power into a z-score: "how many standard deviations above this
    electrode's own resting level is it right now?"

    Why bother: electrode 7 might sit right next to a big neuron and read
    3x louder than electrode 41 at all times. Without normalisation, electrode
    7's preferred direction would hijack every decode. This puts all 64
    channels on equal footing.

    The first 250 ms are assumed to be rest — no movement has started yet.
    """
    n_base = int(baseline_end_s * FS)
    base_mean = power[:, :n_base].mean(axis=1, keepdims=True)
    base_std = power[:, :n_base].std(axis=1, keepdims=True) + 1e-9  # avoid /0

    return (power - base_mean) / base_std


def decode_position(signals, gain=0.12, deadzone=1.5):
    """
    THE CORE BMI STEP: population vector decoding.

    For every millisecond, each electrode votes for its own preferred
    direction, and the strength of its vote is how excited it currently is.
    Sum the votes -> a 2D arrow = intended velocity. Add up velocity over
    time -> position of the robotic limb.

    Parameters
    ----------
    gain : scales neural units into arm units (cm per z-score per second).
           In a real BMI this is calibrated per user, per session.
    deadzone : z-scores below this are treated as zero. Stops the arm from
           slowly wandering off during rest just because noise never averages
           to exactly zero. Real systems do the same thing.

    Returns
    -------
    position : (2000, 2) array of X,Y in centimetres
    velocity : (2000, 2) array of X,Y speed
    activity : (64, 2000) normalised excitement, kept for plotting
    """
    # Steps 1-3: clean, measure, normalise.
    filtered = high_pass(signals)
    power = spiking_power(filtered)
    activity = normalise_to_baseline(power)

    # Apply the deadzone: quiet channels contribute nothing.
    votes = np.where(activity > deadzone, activity - deadzone, 0.0)

    # Step 4: each electrode's preferred direction as a unit arrow (x, y).
    dir_x = np.cos(preferred_dirs)[:, None]        # shape (64, 1)
    dir_y = np.sin(preferred_dirs)[:, None]

    # Weighted sum across all 64 electrodes -> one arrow per time point.
    vel_x = gain * np.sum(votes * dir_x, axis=0)   # shape (2000,)
    vel_y = gain * np.sum(votes * dir_y, axis=0)
    velocity = np.column_stack([vel_x, vel_y])

    # Integrate velocity into position. cumsum * DT is the numerical version
    # of "distance = speed x time", accumulated step by step.
    position = np.cumsum(velocity, axis=0) * DT

    return position, velocity, activity


# =============================================================================
# SECTION 3 — VISUALISE
# =============================================================================

def make_figure(signals, position, outfile):
    """
    Left panel  : raw voltage traces from a handful of electrodes.
    Right panel : the path the robotic limb traced out, coloured by time.
    """
    fig, (ax_raw, ax_path) = plt.subplots(1, 2, figsize=(15, 6.5))

    # ---------------- LEFT: raw electrode voltages -------------------------
    # Plotting all 64 traces at once is unreadable, so show 8 spread across
    # the array. Each is pushed up by a fixed offset so they stack neatly —
    # the y-axis is therefore "which electrode", not absolute voltage.
    shown = np.linspace(0, N_CHANNELS - 1, 8, dtype=int)
    offset_step = 320   # microvolts of vertical space per trace

    for i, ch in enumerate(shown):
        ax_raw.plot(time, signals[ch] + i * offset_step, linewidth=0.55,
                    color=plt.cm.viridis(i / len(shown)))

    # Shade each attempted movement so the bursts are easy to spot.
    for start, dur, direction in MOVEMENTS:
        ax_raw.axvspan(start, start + dur, color="crimson", alpha=0.10)
        ax_raw.text(start + dur / 2, -offset_step * 0.75,
                    f"{np.rad2deg(direction):.0f}°",
                    ha="center", color="crimson", fontsize=9)

    ax_raw.set_yticks([i * offset_step for i in range(len(shown))])
    ax_raw.set_yticklabels([f"ch {c}" for c in shown], fontsize=9)
    ax_raw.set_xlabel("Time (seconds)")
    ax_raw.set_title("Raw motor cortex recording\n"
                     "(8 of 64 electrodes; shaded bands = attempted movements)",
                     fontsize=11)
    ax_raw.set_xlim(0, DURATION)
    ax_raw.spines[["top", "right"]].set_visible(False)

    # ---------------- RIGHT: decoded limb trajectory -----------------------
    # Draw the path as many short segments so each can carry its own colour,
    # which encodes when in the 2 seconds that piece of the path happened.
    points = position.reshape(-1, 1, 2)
    segments = np.concatenate([points[:-1], points[1:]], axis=1)

    lc = LineCollection(segments, cmap="plasma", linewidth=2.4)
    lc.set_array(time[:-1])
    ax_path.add_collection(lc)

    ax_path.plot(*position[0], "o", color="green", markersize=10,
                 label="start (0.0 s)")
    ax_path.plot(*position[-1], "s", color="black", markersize=9,
                 label="end (2.0 s)")

    # Reference arrows showing what each attempted movement *should* look like,
    # so you can eyeball how faithful the decode was.
    for start, dur, direction in MOVEMENTS:
        idx = int(start * FS)
        ax_path.annotate(
            "", xy=(position[idx, 0] + 1.6 * np.cos(direction),
                    position[idx, 1] + 1.6 * np.sin(direction)),
            xytext=position[idx],
            arrowprops=dict(arrowstyle="->", color="gray",
                            linestyle="--", linewidth=1.3),
        )

    cbar = fig.colorbar(lc, ax=ax_path, pad=0.02)
    cbar.set_label("Time (seconds)")

    ax_path.set_xlabel("X position (cm)")
    ax_path.set_ylabel("Y position (cm)")
    ax_path.set_title("Decoded robotic limb path\n"
                      "(dashed arrows = intended direction of each movement)",
                      fontsize=11)
    ax_path.axhline(0, color="lightgray", linewidth=0.8, zorder=0)
    ax_path.axvline(0, color="lightgray", linewidth=0.8, zorder=0)
    ax_path.autoscale_view()
    ax_path.set_aspect("equal", adjustable="datalim")
    ax_path.legend(loc="best", fontsize=9)
    ax_path.spines[["top", "right"]].set_visible(False)

    fig.suptitle("Simulated brain-machine interface: 64-electrode motor cortex "
                 "array driving a 2D robotic limb", fontsize=13, y=0.99)
    fig.tight_layout()
    fig.savefig(outfile, dpi=150)
    print(f"Figure saved to {outfile}")


# =============================================================================
# RUN IT
# =============================================================================

if __name__ == "__main__":
    print("Generating 2 s of 64-channel neural data...")
    signals = generate_recording()
    print(f"  recording shape: {signals.shape}  (electrodes x samples)")
    print(f"  voltage range:   {signals.min():.0f} to {signals.max():.0f} uV")

    print("Decoding intended movement...")
    position, velocity, activity = decode_position(signals)
    print(f"  final limb position: "
          f"X={position[-1, 0]:.2f} cm, Y={position[-1, 1]:.2f} cm")
    print(f"  peak speed:          "
          f"{np.max(np.linalg.norm(velocity, axis=1)):.2f} cm/s")

    make_figure(signals, position, "bmi_simulation.png")
    plt.savefig("bmi_simulation.png", dpi=300, bbox_inches="tight")
plt.show()


Generating 2 s of 64-channel neural data...
  recording shape: (64, 2000)  (electrodes x samples)
  voltage range:   -161 to 170 uV
Decoding intended movement...
  final limb position: X=0.91 cm, Y=-3.71 cm
  peak speed:          25.44 cm/s
Figure saved to bmi_simulation.png


### 🧭 Activity A.2 — Simulating Brain Imaging (fMRI) Data

> *Copy the prompt below into your preferred AI tool, then paste the generated code into the code cell beneath it.*
>
> "Act as a neuroscience instructor explaining fMRI to a biology student. Write a Python script that builds a simplified simulation of a functional brain imaging dataset. The script should:
> 1. Create a small 3D grid of brain regions (10 x 10 x 10 grid points), each observed across 15 time points — representing a short fMRI recording session.
> 2. Simulate a realistic brain activity pattern where a specific region becomes active (increases its signal) and then returns to baseline, mimicking what happens when a participant performs a task.
> 3. Add a simple head movement correction step: introduce a small artificial shift in the data across time points, then show how a correction routine detects and compensates for this.
> 4. Display a plot showing how the signal changes over time in an active brain region compared to an inactive one.
> Use plain-language comments to explain what each step represents biologically."

In [6]:
# 💻 Activity A.2 — Paste your AI-generated fMRI Simulation code here:
import numpy as np
import matplotlib.pyplot as plt

# ============================================================
# Simplified fMRI simulation
# ============================================================
# An fMRI scanner measures changes in blood oxygenation (BOLD
# signal) across many small 3-D locations in the brain.
#
# Here we create:
#   - a 10 x 10 x 10 brain grid = 1,000 locations ("voxels")
#   - 15 time points
#   - one region that becomes active during a task
#   - background biological/scanner noise
#   - artificial head movement
#   - a simple motion-correction procedure
#
# This is an educational simulation, not a model of real fMRI
# physics or a clinical preprocessing pipeline.
# ============================================================

# Use a fixed random seed so the simulation gives the same
# results each time the program is run.
rng = np.random.default_rng(42)

# ------------------------------------------------------------
# 1. Define the simulated brain
# ------------------------------------------------------------

grid_size = 10
n_timepoints = 15

# A 10 x 10 x 10 grid contains 1,000 simulated brain locations.
# In real fMRI, these locations would be called voxels.
shape = (grid_size, grid_size, grid_size)

# Start with a baseline BOLD signal of 100 arbitrary units.
# The absolute number is not important here; we mainly care
# about changes in the signal over time.
baseline = 100.0

brain = np.full(
    shape + (n_timepoints,),
    baseline,
    dtype=float
)

# ------------------------------------------------------------
# 2. Add normal background variation
# ------------------------------------------------------------

# Real fMRI measurements contain noise from many sources:
# scanner noise, physiological fluctuations, and other effects.
#
# We add small random fluctuations around the baseline.
noise = rng.normal(
    loc=0,
    scale=1.5,
    size=brain.shape
)

brain += noise


# ------------------------------------------------------------
# 3. Create an "active" brain region
# ------------------------------------------------------------

# Pick the center of the brain as our simulated task-related
# region. In a real experiment, this could represent a small
# portion of motor cortex, visual cortex, etc.
center = np.array([5, 5, 5])

# Create a spherical region around the center.
# Locations inside this radius will respond to the task.
radius = 2

x, y, z = np.indices(shape)

distance = np.sqrt(
    (x - center[0]) ** 2 +
    (y - center[1]) ** 2 +
    (z - center[2]) ** 2
)

active_region = distance <= radius


# ------------------------------------------------------------
# 4. Simulate a task-related BOLD response
# ------------------------------------------------------------

# The participant is assumed to perform a task during the
# middle portion of the recording.
#
# The numbers represent arbitrary BOLD percentage-like changes:
#   baseline -> normal activity
#   increase -> task-related brain activation
#   decrease -> return toward baseline
#
# With only 15 time points, this is intentionally simplified.

task_response = np.array([
    0.0, 0.0, 0.5, 1.5, 3.0,
    5.0, 6.0, 5.0, 3.5, 2.0,
    1.0, 0.3, 0.0, 0.0, 0.0
])

# Add the task response to every voxel in the active region.
for t in range(n_timepoints):
    brain[:, :, :, t][active_region] += task_response[t]


# ------------------------------------------------------------
# 5. Choose an inactive comparison region
# ------------------------------------------------------------

# We will compare the active region with one voxel outside it.
# This voxel does not receive the artificial task activation.
inactive_voxel = (1, 1, 1)

# The center voxel gives us a representative measurement from
# the active brain region.
active_voxel = tuple(center)


# ------------------------------------------------------------
# 6. Introduce artificial head movement
# ------------------------------------------------------------

# In an actual MRI scanner, even small head movements can cause
# the brain to appear to shift from one time point to another.
#
# Here we simulate this by shifting the entire 3-D brain volume.
#
# The shifts are measured in grid points (voxels).
movement_shifts = [
    (0, 0, 0),
    (0, 0, 0),
    (1, 0, 0),
    (1, 0, 0),
    (1, 1, 0),
    (1, 1, 0),
    (0, 1, 0),
    (0, 1, 0),
    (0, 0, 0),
    (0, 0, 0),
    (-1, 0, 0),
    (-1, 0, 0),
    (0, 0, 0),
    (0, 0, 0),
    (0, 0, 0),
]

# Create a new array to hold the motion-corrupted data.
moved_brain = np.empty_like(brain)

for t, shift in enumerate(movement_shifts):
    # np.roll moves the simulated brain volume.
    # This is a simple way to imitate head motion.
    moved_brain[:, :, :, t] = np.roll(
        brain[:, :, :, t],
        shift=shift,
        axis=(0, 1, 2)
    )


# ------------------------------------------------------------
# 7. Simple head-motion correction
# ------------------------------------------------------------

# We pretend that a motion-estimation algorithm has detected
# the shifts we introduced above.
#
# To undo a shift, we apply the opposite shift.
corrected_brain = np.empty_like(moved_brain)

for t, shift in enumerate(movement_shifts):

    # Reverse the detected movement.
    reverse_shift = tuple(-value for value in shift)

    corrected_brain[:, :, :, t] = np.roll(
        moved_brain[:, :, :, t],
        shift=reverse_shift,
        axis=(0, 1, 2)
    )


# ------------------------------------------------------------
# 8. Extract the signals we want to compare
# ------------------------------------------------------------

# Signal from the active region after motion correction.
active_signal = np.array([
    corrected_brain[active_voxel[0],
                    active_voxel[1],
                    active_voxel[2],
                    t]
    for t in range(n_timepoints)
])

# Signal from an inactive location.
inactive_signal = np.array([
    corrected_brain[inactive_voxel[0],
                    inactive_voxel[1],
                    inactive_voxel[2],
                    t]
    for t in range(n_timepoints)
])

# Time is represented here simply as 15 consecutive measurements.
time = np.arange(1, n_timepoints + 1)


# ------------------------------------------------------------
# 9. Print a simple summary
# ------------------------------------------------------------

print("Simplified fMRI simulation")
print("--------------------------")
print(f"Brain grid: {grid_size} x {grid_size} x {grid_size}")
print(f"Number of simulated voxels: {grid_size ** 3}")
print(f"Number of time points: {n_timepoints}")
print()
print("Simulated head-motion shifts:")
for t, shift in enumerate(movement_shifts, start=1):
    print(f"  Time {t:2d}: shift = {shift}")

print()
print(
    "Peak active-region signal:",
    f"{active_signal.max():.2f} arbitrary units"
)

print(
    "Inactive-region average signal:",
    f"{inactive_signal.mean():.2f} arbitrary units"
)


# ------------------------------------------------------------
# 10. Plot the fMRI signal over time
# ------------------------------------------------------------

plt.figure(figsize=(10, 6))

plt.plot(
    time,
    active_signal,
    marker="o",
    linewidth=2,
    color="crimson",
    label="Active brain region"
)

plt.plot(
    time,
    inactive_signal,
    marker="o",
    linewidth=2,
    color="steelblue",
    label="Inactive brain region"
)

# The shaded region indicates approximately when the simulated
# participant is performing the task.
plt.axvspan(
    3,
    10,
    color="orange",
    alpha=0.15,
    label="Simulated task period"
)

plt.axhline(
    baseline,
    color="gray",
    linestyle="--",
    alpha=0.7,
    label="Baseline"
)

plt.xlabel("Time point")
plt.ylabel("BOLD signal (arbitrary units)")

plt.title(
    "Simplified fMRI: Task-Related Brain Activation"
)

plt.legend()
plt.grid(alpha=0.25)
plt.tight_layout()

# Save the figure as an image file.
plt.savefig(
    "fmri_simulation.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

print()
print("Figure saved as: fmri_simulation.png")


Simplified fMRI simulation
--------------------------
Brain grid: 10 x 10 x 10
Number of simulated voxels: 1000
Number of time points: 15

Simulated head-motion shifts:
  Time  1: shift = (0, 0, 0)
  Time  2: shift = (0, 0, 0)
  Time  3: shift = (1, 0, 0)
  Time  4: shift = (1, 0, 0)
  Time  5: shift = (1, 1, 0)
  Time  6: shift = (1, 1, 0)
  Time  7: shift = (0, 1, 0)
  Time  8: shift = (0, 1, 0)
  Time  9: shift = (0, 0, 0)
  Time 10: shift = (0, 0, 0)
  Time 11: shift = (-1, 0, 0)
  Time 12: shift = (-1, 0, 0)
  Time 13: shift = (0, 0, 0)
  Time 14: shift = (0, 0, 0)
  Time 15: shift = (0, 0, 0)

Peak active-region signal: 109.37 arbitrary units
Inactive-region average signal: 99.65 arbitrary units

Figure saved as: fmri_simulation.png


### ✍️ Part A Reflection
*Double-click this cell to write your response.*

* **What I observed:** ChatGPT and Claude yield different results in terms of number of channels and timeline. Prompt needs edits.
* **Connecting to key concepts:** Activities A.1 and A.2 demonstrate **Concepts 1, 3, 4, 5, 6, and 7** from your Reference Guide. In your own words, explain what a "decoder" does and why correcting for head movement matters before analyzing brain imaging data - decoder converts raw signal into presumably interpretable data; head mvmt correction allow for NOT doing misinterpretind the data.

---

## 💻 Part B: How Computers Store and Organize Research Data
### Topic: Working Memory, Permanent Storage, and Keeping Workflow Stages Separate

This section explores the practical computing concepts that affect how quickly and reliably a research workflow runs.

### 🧭 Activity B.1 — Comparing Working Memory and Permanent Storage

> *Copy the prompt below into your preferred AI tool, then paste the generated code into the code cell beneath it.*
>
> "Act as a computing instructor helping a biology researcher understand how computers handle data. Write a Python script that demonstrates the difference between working memory (RAM) and permanent storage (hard drive). The script should:
> 1. Measure how long it takes to read and write a large array of one million numbers directly in working memory versus saving the same data to a temporary file on disk and reading it back.
> 2. Create a loop that gradually increases the size of a data array, reporting how much working memory is being used at each step, and printing a clear warning message when memory is getting full — stopping safely before the computer runs out.
> 3. Print a clear summary comparing the speed of working memory versus disk storage.
> Use plain-language comments throughout explaining what is happening and why it matters for research workflows."

In [8]:
# 💻 Activity B.1 — Paste your AI-generated Working Memory vs. Storage code here:
import os
import time
import tempfile

import numpy as np


# ============================================================
# RAM vs. permanent storage demonstration
# ============================================================
#
# RAM (working memory):
#   Data is held directly in the computer's memory while a program
#   is running. RAM is very fast, but it is temporary.
#
# Disk storage:
#   Data saved to an SSD or hard drive remains there after the
#   program finishes. It is persistent, but normally slower to
#   access than RAM.
#
# This example measures the difference using one million numbers.
#
# IMPORTANT:
# This is a demonstration, not a precise benchmark. Actual speeds
# depend heavily on your computer, operating system, storage device,
# background programs, and file system.
# ============================================================


# ------------------------------------------------------------
# Optional memory-monitoring package
# ------------------------------------------------------------
#
# psutil lets us ask the operating system how much RAM is available.
# If it isn't installed, the rest of the demonstration still works.
try:
    import psutil
    HAVE_PSUTIL = True
except ImportError:
    HAVE_PSUTIL = False


# ------------------------------------------------------------
# Helper functions
# ------------------------------------------------------------

def format_bytes(number):
    """Turn a number of bytes into an easier-to-read value."""

    units = ["B", "KB", "MB", "GB", "TB"]

    value = float(number)

    for unit in units:
        if value < 1024:
            return f"{value:.2f} {unit}"
        value /= 1024

    return f"{value:.2f} PB"


def get_memory_info():
    """
    Return current process RAM use and available system RAM.

    psutil gives us a convenient cross-platform way to ask the
    operating system about memory.
    """

    if not HAVE_PSUTIL:
        return None, None

    process = psutil.Process(os.getpid())

    # RSS = Resident Set Size: approximately how much physical RAM
    # is currently being used by this Python process.
    process_memory = process.memory_info().rss

    # available = RAM that the operating system can make available
    # without needing to heavily rely on swapping.
    available_memory = psutil.virtual_memory().available

    return process_memory, available_memory


# ------------------------------------------------------------
# Part 1: Create one million numbers in RAM
# ------------------------------------------------------------

N = 1_000_000

print("=" * 60)
print("PART 1: WORKING MEMORY (RAM)")
print("=" * 60)

print(f"\nCreating an array containing {N:,} numbers...")

# float64 means each number occupies 8 bytes.
# One million numbers therefore require approximately 8 MB
# just for the numerical data itself.
start = time.perf_counter()

ram_array = np.arange(
    N,
    dtype=np.float64
)

ram_create_time = time.perf_counter() - start

print(f"Array size:       {format_bytes(ram_array.nbytes)}")
print(f"Creation time:    {ram_create_time:.6f} seconds")

process_memory, available_memory = get_memory_info()

if process_memory is not None:
    print(f"Python RAM use:   {format_bytes(process_memory)}")
    print(f"RAM available:    {format_bytes(available_memory)}")
else:
    print("Detailed system RAM information requires psutil.")


# ------------------------------------------------------------
# Part 2: Read and write the array while it is in RAM
# ------------------------------------------------------------

print("\nTesting RAM read/write speed...")

# WRITE:
# Change every value in the array.
start = time.perf_counter()

ram_array[:] = ram_array * 2.0

ram_write_time = time.perf_counter() - start


# READ:
# Calculate a summary value from every element.
# The program must actually use the data so that this represents
# a real computational read rather than an unused variable.
start = time.perf_counter()

ram_sum = np.sum(ram_array)

ram_read_time = time.perf_counter() - start

print(f"RAM write time:   {ram_write_time:.6f} seconds")
print(f"RAM read time:    {ram_read_time:.6f} seconds")
print(f"RAM data sum:     {ram_sum:.0f}")


# ------------------------------------------------------------
# Part 3: Save the same data to disk
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("PART 2: DISK STORAGE")
print("=" * 60)

# NamedTemporaryFile gives us a temporary file that the operating
# system can remove when we are finished.
#
# delete=False makes this work more consistently across operating
# systems, including Windows.
temp_file = tempfile.NamedTemporaryFile(
    suffix=".bin",
    delete=False
)

temp_filename = temp_file.name
temp_file.close()

try:

    # --------------------------------------------------------
    # WRITE TO DISK
    # --------------------------------------------------------

    print("\nSaving the one-million-number array to disk...")

    start = time.perf_counter()

    # tofile writes the binary contents of the NumPy array
    # directly to the temporary file.
    ram_array.tofile(temp_filename)

    disk_write_time = time.perf_counter() - start

    print(f"Disk write time:  {disk_write_time:.6f} seconds")
    print(f"File size:        {format_bytes(os.path.getsize(temp_filename))}")


    # --------------------------------------------------------
    # READ FROM DISK
    # --------------------------------------------------------

    print("\nReading the array back from disk...")

    start = time.perf_counter()

    disk_array = np.fromfile(
        temp_filename,
        dtype=np.float64
    )

    disk_read_time = time.perf_counter() - start

    print(f"Disk read time:   {disk_read_time:.6f} seconds")
    print(f"Numbers recovered:{len(disk_array):,}")
    print(f"Data sum:         {np.sum(disk_array):.0f}")


    # --------------------------------------------------------
    # Compare the results
    # --------------------------------------------------------

    if disk_write_time > 0:
        write_speed_ratio = disk_write_time / max(ram_write_time, 1e-12)
    else:
        write_speed_ratio = float("inf")

    if disk_read_time > 0:
        read_speed_ratio = disk_read_time / max(ram_read_time, 1e-12)
    else:
        read_speed_ratio = float("inf")

    print("\n" + "=" * 60)
    print("SPEED SUMMARY")
    print("=" * 60)

    print(f"\nRAM write:   {ram_write_time:.6f} seconds")
    print(f"Disk write:  {disk_write_time:.6f} seconds")

    print(f"\nRAM read:    {ram_read_time:.6f} seconds")
    print(f"Disk read:   {disk_read_time:.6f} seconds")

    print(
        f"\nDisk/RAM write-time ratio: "
        f"{write_speed_ratio:.1f}x"
    )

    print(
        f"Disk/RAM read-time ratio:  "
        f"{read_speed_ratio:.1f}x"
    )

    print("\nInterpretation:")
    print(
        "RAM is designed for fast, active computation, while "
        "disk storage is designed for keeping data."
    )
    print(
        "For research workflows, this is why large datasets are "
        "often loaded into RAM when being actively analyzed, then "
        "saved back to disk for long-term storage."
    )


    # --------------------------------------------------------
    # Part 4: Gradually increase the RAM workload
    # --------------------------------------------------------

    print("\n" + "=" * 60)
    print("PART 3: GRADUALLY USING MORE RAM")
    print("=" * 60)

    print(
        "\nThe program will create progressively larger arrays."
    )

    if not HAVE_PSUTIL:
        print(
            "\nNOTE: Install psutil for detailed system-memory "
            "monitoring:"
        )
        print("    pip install psutil")
        print(
            "\nWithout psutil, the script uses a conservative "
            "array-size safety limit."
        )

    # We deliberately stop well before trying to fill the entire
    # computer's RAM. This makes the demonstration safer.
    #
    # Each float64 number uses 8 bytes.
    MAX_ARRAY_BYTES = 1 * 1024**3  # 1 GB maximum for this demo

    # If psutil is available, use only a fraction of currently
    # available RAM rather than approaching the system limit.
    if HAVE_PSUTIL:
        available = psutil.virtual_memory().available

        # Never allow the demonstration to consume more than
        # 25% of currently available RAM.
        safe_limit = min(
            MAX_ARRAY_BYTES,
            int(available * 0.25)
        )
    else:
        safe_limit = MAX_ARRAY_BYTES

    print(
        f"\nSafety limit for this demonstration: "
        f"{format_bytes(safe_limit)}"
    )

    # Start at 100,000 numbers and approximately double the size
    # at every step.
    current_size = 100_000

    arrays = []

    while True:

        # Calculate how much RAM this particular NumPy array needs.
        required_bytes = current_size * np.dtype(np.float64).itemsize

        # Check the safety limit BEFORE allocating the array.
        if required_bytes > safe_limit:
            print(
                "\nSAFE STOP: The next array would exceed the "
                "memory limit chosen for this demonstration."
            )
            break

        # If psutil is available, check current available RAM too.
        if HAVE_PSUTIL:
            available = psutil.virtual_memory().available

            # We stop if less than 10% of the currently available
            # memory remains. This helps prevent accidentally
            # overwhelming the computer.
            if available < safe_limit * 0.10:
                print(
                    "\nWARNING: Available RAM is getting low."
                )
                print(
                    "Stopping before the computer runs out "
                    "of working memory."
                )
                break

        # Allocate the next array.
        print(
            f"\nCreating {current_size:,} numbers..."
        )

        try:
            new_array = np.ones(
                current_size,
                dtype=np.float64
            )

            arrays.append(new_array)

        except MemoryError:
            # If the operating system refuses the allocation,
            # stop safely instead of continuing.
            print(
                "\nWARNING: The operating system could not "
                "allocate this array."
            )
            print(
                "Stopping safely before requesting more memory."
            )
            break

        # Report the amount of memory represented by this array.
        print(
            f"  This array uses: "
            f"{format_bytes(new_array.nbytes)}"
        )

        # Report the memory used by the Python process when possible.
        process_memory, available_memory = get_memory_info()

        if process_memory is not None:
            print(
                f"  Python process RAM: "
                f"{format_bytes(process_memory)}"
            )

            print(
                f"  System RAM available: "
                f"{format_bytes(available_memory)}"
            )

            # Give the researcher a clear warning before memory
            # becomes dangerously low.
            if available_memory < 1 * 1024**3:
                print(
                    "  WARNING: Less than 1 GB of RAM remains "
                    "available!"
                )

                print(
                    "  Stopping the memory-growth experiment."
                )
                break

        # Double the size for the next iteration.
        current_size *= 2

    # --------------------------------------------------------
    # Clean up the RAM experiment
    # --------------------------------------------------------

    print("\nCleaning up demonstration arrays...")

    # Removing references allows Python to release the arrays.
    arrays.clear()

finally:

    # --------------------------------------------------------
    # Remove the temporary disk file
    # --------------------------------------------------------

    # We don't want this demonstration file to remain on the
    # researcher's computer after the program finishes.
    if os.path.exists(temp_filename):
        os.remove(temp_filename)

    print("\nTemporary disk file removed.")

print("\n" + "=" * 60)
print("DEMONSTRATION COMPLETE")
print("=" * 60)

print(
    "\nKey idea:"
    "\n  RAM = fast working space for active analysis"
    "\n  Disk = slower but persistent space for storing data"
)

print(
    "\nIn a research workflow, a large dataset might live on "
    "disk for long-term storage, while the portion currently "
    "being analyzed is loaded into RAM."
)


PART 1: WORKING MEMORY (RAM)

Creating an array containing 1,000,000 numbers...
Array size:       7.63 MB
Creation time:    0.004454 seconds
Python RAM use:   198.04 MB
RAM available:    11.32 GB

Testing RAM read/write speed...
RAM write time:   0.003743 seconds
RAM read time:    0.001180 seconds
RAM data sum:     999999000000

PART 2: DISK STORAGE

Saving the one-million-number array to disk...
Disk write time:  0.007010 seconds
File size:        7.63 MB

Reading the array back from disk...
Disk read time:   0.002560 seconds
Numbers recovered:1,000,000
Data sum:         999999000000

SPEED SUMMARY

RAM write:   0.003743 seconds
Disk write:  0.007010 seconds

RAM read:    0.001180 seconds
Disk read:   0.002560 seconds

Disk/RAM write-time ratio: 1.9x
Disk/RAM read-time ratio:  2.2x

Interpretation:
RAM is designed for fast, active computation, while disk storage is designed for keeping data.
For research workflows, this is why large datasets are often loaded into RAM when being active

### 🧭 Activity B.2 — Keeping Workflow Stages Independent

> *Copy the prompt below into your preferred AI tool, then paste the generated code into the code cell beneath it.*
>
> "Act as a research software instructor. Write a Python script that demonstrates why scientific workflows should be organized into separate, independent stages — rather than one large block of code. The script should:
> 1. Stage 1 (Obtaining Data): A function that loads or generates a set of recorded sensor measurements. This function should not perform any calculations or create any plots.
> 2. Stage 2 (Analyzing Data): A function that receives the data from Stage 1 and calculates summary statistics (mean, standard deviation, range). This function should not load data or create plots.
> 3. Stage 3 (Visualizing Results): A function that takes the results from Stage 2 and creates a clean summary plot.
> Show clearly how these three stages pass information to each other, and explain in comments what would go wrong if all three were mixed together in a single block of code."

In [9]:
# 💻 Activity B.2 — Paste your AI-generated Separated Workflow Stages code here:
import numpy as np
import matplotlib.pyplot as plt


# ============================================================
# Stage 1: OBTAINING DATA
# ============================================================

def obtain_data(n_measurements=500):
    """
    Generate simulated sensor measurements.

    In a real research project, this function could instead:
      - read a CSV file,
      - load an instrument's output,
      - read a database,
      - or receive measurements from a laboratory device.

    IMPORTANT:
    This function ONLY obtains the data.
    It does not calculate statistics and does not make plots.

    Returns
    -------
    numpy.ndarray
        One-dimensional array containing sensor measurements.
    """

    print("Stage 1: Obtaining sensor data...")

    # Use a fixed random seed so that the example produces
    # the same simulated measurements each time.
    rng = np.random.default_rng(42)

    # Imagine these are measurements from a biological sensor.
    # For example, the sensor might measure temperature,
    # fluorescence, voltage, or some other experimental variable.
    #
    # The average measurement is approximately 25 units,
    # with normal biological/measurement variation.
    data = rng.normal(
        loc=25.0,
        scale=2.0,
        size=n_measurements
    )

    # Return the data to whoever called this function.
    #
    # Notice that nothing has been calculated or plotted here.
    return data


# ============================================================
# Stage 2: ANALYZING DATA
# ============================================================

def analyze_data(data):
    """
    Calculate summary statistics from already-loaded data.

    IMPORTANT:
    This function receives data as an argument.
    It does NOT load files, generate measurements, or create plots.

    Parameters
    ----------
    data : numpy.ndarray
        Measurements produced by Stage 1.

    Returns
    -------
    dict
        Dictionary containing the mean, standard deviation,
        minimum, maximum, and range.
    """

    print("Stage 2: Analyzing sensor data...")

    # Calculate the average measurement.
    mean = np.mean(data)

    # Standard deviation describes how spread out the
    # measurements are around the mean.
    standard_deviation = np.std(data)

    # Find the smallest and largest measurements.
    minimum = np.min(data)
    maximum = np.max(data)

    # Range is the distance from the smallest to largest value.
    data_range = maximum - minimum

    # Put the results into a dictionary.
    #
    # This dictionary becomes the output of Stage 2 and
    # the input to Stage 3.
    results = {
        "mean": mean,
        "standard_deviation": standard_deviation,
        "minimum": minimum,
        "maximum": maximum,
        "range": data_range
    }

    return results


# ============================================================
# Stage 3: VISUALIZING RESULTS
# ============================================================

def visualize_results(results):
    """
    Create a plot from the summary statistics.

    IMPORTANT:
    This function receives results from Stage 2.
    It does NOT load data or perform the original analysis.

    Parameters
    ----------
    results : dict
        Summary statistics generated by analyze_data().
    """

    print("Stage 3: Visualizing results...")

    # Extract the statistics produced by Stage 2.
    labels = [
        "Mean",
        "Std. Dev.",
        "Range"
    ]

    values = [
        results["mean"],
        results["standard_deviation"],
        results["range"]
    ]

    # Create a simple summary plot.
    fig, ax = plt.subplots(figsize=(8, 5))

    bars = ax.bar(
        labels,
        values,
        color=["steelblue", "darkorange", "seagreen"]
    )

    ax.set_ylabel("Value")
    ax.set_title("Summary of Sensor Measurements")
    ax.grid(
        axis="y",
        alpha=0.25
    )

    # Write the numerical value above each bar.
    for bar, value in zip(bars, values):
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height(),
            f"{value:.2f}",
            ha="center",
            va="bottom"
        )

    plt.tight_layout()

    # Save the figure so it can become part of a research report
    # or electronic lab notebook.
    plt.savefig(
        "sensor_summary.png",
        dpi=300,
        bbox_inches="tight"
    )

    plt.show()


# ============================================================
# MAIN WORKFLOW
# ============================================================

def main():

    print("=" * 60)
    print("MODULAR SCIENTIFIC DATA WORKFLOW")
    print("=" * 60)

    # --------------------------------------------------------
    # Stage 1 -> Stage 2
    # --------------------------------------------------------
    #
    # obtain_data() produces the measurements.
    # Those measurements are passed directly into analyze_data().
    #
    # Data flow:
    #
    #       obtain_data()
    #             |
    #             v
    #        sensor_data
    #             |
    #             v
    #       analyze_data()
    #
    sensor_data = obtain_data()

    print(f"  Received {len(sensor_data)} measurements.")
    print()

    # --------------------------------------------------------
    # Stage 2 -> Stage 3
    # --------------------------------------------------------
    #
    # analyze_data() receives the measurements and returns
    # a compact set of results.
    #
    # Data flow:
    #
    #       sensor_data
    #             |
    #             v
    #       analyze_data()
    #             |
    #             v
    #          results
    #             |
    #             v
    #       visualize_results()
    #
    results = analyze_data(sensor_data)

    print()
    print("Summary statistics:")
    print(f"  Mean:              {results['mean']:.2f}")
    print(
        f"  Standard deviation:"
        f" {results['standard_deviation']:.2f}"
    )
    print(f"  Minimum:            {results['minimum']:.2f}")
    print(f"  Maximum:            {results['maximum']:.2f}")
    print(f"  Range:              {results['range']:.2f}")
    print()

    # Pass only the analysis results to the visualization stage.
    visualize_results(results)

    print()
    print("=" * 60)
    print("WORKFLOW COMPLETE")
    print("=" * 60)


# Run the workflow when this file is executed.
if __name__ == "__main__":
    main()


MODULAR SCIENTIFIC DATA WORKFLOW
Stage 1: Obtaining sensor data...
  Received 500 measurements.

Stage 2: Analyzing sensor data...

Summary statistics:
  Mean:              24.97
  Standard deviation: 1.92
  Minimum:            19.87
  Maximum:            30.83
  Range:              10.96

Stage 3: Visualizing results...

WORKFLOW COMPLETE


### ✍️ Part B Reflection
*Double-click this cell to write your response.*

* **What I observed:** Based on Activity B.1, how much faster was working memory (RAM) compared to saving and loading from disk? - quite a bit. What happened as the data size grew? -RAM also got used faster.
* **Connecting to key concepts:** Activities B.1 and B.2 demonstrate **Concepts 26, 29, 36, 39, and 41** from your Reference Guide. Explain in plain language why a researcher should keep the data-loading stage of a workflow completely separate from the visualization stage. - If everything were placed in one giant block, changing the data source could accidentally affect the calculations or plotting. Debugging would also be harder because data loading, analysis, and visualization would be intertwined. A modular workflow lets a researcher test each stage independently and reuse the same analysis code across many experiments.
---

## 📑 What Comes Next

The next four pre-workshop notebooks build directly on the foundations you have established here. Notebook 2 introduces working with datasets from many participants at once. Notebook 3 covers signal cleaning and filtering. Notebook 4 introduces pattern recognition and machine learning. Notebook 5 brings everything together into a complete integrated workflow.


---

## 📖 Reference Guide: 50 Key Concepts for Neuroscience Workflows

Keep this reference guide handy throughout all five pre-workshop notebooks and during the onsite labs. These concepts form the foundation of the workshop's hands-on activities.

### 🧠 Neuroscience & Research Applications (Concepts 1–25)
1. **Brain-Machine Interface (BMI):** A system that connects the brain directly to an external device — bypassing damaged nerves or muscles — so that brain signals can control a robotic limb, cursor, or other tool.
2. **Non-Invasive (EEG) vs. Invasive (Intracortical) Sensors:** Scalp electrodes (EEG) are easy to use but pick up blurry, averaged signals through the skull. Implanted microelectrode arrays record from individual neurons with much greater precision, but require surgery.
3. **Signal Delay (Latency):** The time gap between a brain signal being recorded and a device responding to it. Delays longer than about 50–100 milliseconds feel unnatural to the user of a prosthetic device.
4. **Decoder:** A mathematical or statistical model that translates continuous brain signal patterns into a useful output — such as movement direction, cursor position, or speech intent.
5. **fMRI:** Functional Magnetic Resonance Imaging — a brain scanning method that measures blood oxygen levels as a stand-in for neural activity. When neurons become active, they consume more oxygen, causing a detectable change in the local blood signal.
6. **Voxel:** A small 3D cube of brain tissue in an fMRI image — the brain-imaging equivalent of a pixel. Each voxel contains millions of neurons.
7. **Head Movement Correction:** A processing step that aligns brain scan images across time, compensating for small movements the participant made during the recording session.
8. **Open-Loop vs. Closed-Loop Systems:** An open-loop device follows a fixed program regardless of what the user's brain is doing. A closed-loop device continuously reads incoming signals and adjusts its output in real time based on what it detects.
9. **Deep Brain Stimulation (DBS):** A treatment for conditions like Parkinson's disease in which a small implanted device delivers electrical pulses to specific brain regions to reduce tremor and improve movement.
10. **EMG (Electromyogram):** A recording of the electrical signals produced by muscles when they contract. Often used alongside brain recordings to verify whether a behavioral response actually occurred.
11. **Continuous Recordings vs. Event Markers:** A continuous recording captures an uninterrupted time series of measurements. Event markers are specific timestamps that label when something important happened — such as when a stimulus appeared or a button was pressed.
12. **Signal Drift:** The gradual change in a recorded signal over time, not due to the brain, but due to electrode movement, tissue changes, or electronic drift. Workflows need to account for this to keep analysis accurate over long sessions.
13. **Open Data Repositories:** Publicly accessible online archives (such as DANDI, OpenNeuro, or the Human Connectome Project) where researchers share their raw data so others can analyze or replicate their findings.
14. **Automated Data Download (API):** A method of downloading data directly inside a script using a standardized web link, rather than clicking through a website manually. This makes data access reproducible and audit-able.
15. **Artifact Removal:** The process of identifying and removing unwanted signals — such as electrical noise from the building, muscle movements, or eye blinks — from a raw brain recording before analysis.
16. **Downsampling:** Reducing the number of data points per second in a recording, to save memory and processing time, when the extra detail is not needed for the analysis.
17. **Standard Data Formats (BIDS):** A community-agreed system for naming and organizing brain imaging files so that any researcher or software tool can understand the structure without needing special instructions.
18. **Spectrogram:** A visual display showing how the frequency content of a signal changes over time — useful for seeing when the brain shifts between different rhythmic states such as sleep stages or attention levels.
19. **Nyquist Rule:** A fundamental rule of digital recording: to accurately capture a signal, you must record at least twice as fast as the highest frequency in that signal. Recording too slowly creates false patterns called aliasing.
20. **Local Field Potential (LFP):** An electrical recording that reflects the combined activity of a small cluster of nearby neurons — capturing the general activity level of a local brain region rather than individual cells.
21. **Signal Transfer Function:** A mathematical description of how an input signal is transformed into an output signal by a processing step — useful for predicting what a filter or decoder will do to any given input.
22. **Spike Sorting:** The process of separating a mixed electrical recording from multiple nearby neurons into individual neuron signals, based on the distinct shape of each neuron's electrical discharge.
23. **Machine Learning for Neural Decoding:** Using statistical learning algorithms to automatically find patterns in brain signal data that predict behavior, movement intent, or cognitive state.
24. **Sensory Feedback:** Sending information back to the user of a brain-machine interface — for example, delivering a gentle electrical sensation to the skin to simulate the feeling of touching an object with a prosthetic hand.
25. **Neural Plasticity:** The brain's ability to reorganize its connections over time. Relevant to BMI research because users can learn to control devices more accurately with practice as their brain adapts.

### 💻 Computing & Workflow Fundamentals (Concepts 26–50)
26. **Working Memory (RAM) vs. Permanent Storage:** RAM (working memory) holds data only while the computer is on — it is extremely fast but temporary. The hard drive stores files permanently but is much slower to access.
27. **Processor Core:** A single computing unit inside a central processor (CPU). Most modern computers have multiple cores, allowing several tasks to run at the same time.
28. **CPU vs. GPU:** A CPU handles complex, varied tasks one at a time across a few powerful cores. A GPU handles simple, repetitive tasks across thousands of smaller cores simultaneously — useful for image processing and machine learning.
29. **Memory Overflow:** What happens when a script tries to load more data into working memory (RAM) than the computer has available — causing the program to crash.
30. **Motherboard:** The main circuit board inside a computer that connects all the components — processor, memory, storage, and network — so they can communicate with each other.
31. **Processor Slowdown (Thermal Throttling):** When a processor gets too hot, it automatically slows itself down to prevent damage. This can cause unexpected slowdowns during long analysis runs.
32. **High-Speed Processor Cache:** A small, extremely fast memory area built directly into the processor, used to store frequently needed values so they do not have to be fetched from RAM repeatedly.
33. **Local vs. Cloud Computing:** Running your analysis on the computer in front of you (local) versus running it on a remote server accessed over the internet (cloud). Cloud computing allows access to much more memory and processing power.
34. **Temporary Cloud Workspace:** Cloud computing environments like Google Colab provide a temporary workspace that is automatically cleared when you close the session. Any files you need to keep must be saved to permanent storage before the session ends.
35. **Internet Speed as a Bottleneck:** When downloading large research datasets, the speed of your internet connection often limits how fast data can arrive — regardless of how fast your computer itself is.
36. **Organizing Code into Stages:** Separating a workflow into clearly defined, independent stages — such as one stage for loading data, one for analysis, and one for visualization — makes it much easier to find and fix problems.
37. **Whole Numbers vs. Decimal Numbers in Computing:** Computers store whole numbers (integers) very efficiently. Decimal numbers (floating-point) require more memory and processing time. Choosing the right type for your data can affect both speed and accuracy.
38. **Automated Data Access (API):** A standardized connection that allows one piece of software to request data or services from another automatically — for example, a script that downloads data from a research repository without any manual steps.
39. **Settings Files (JSON/YAML):** Lightweight text files used to store configuration settings, metadata, and parameters for a workflow — making it easy to share, reproduce, or adjust an analysis without changing the code itself.
40. **Processing Delay:** The time between when data arrives in your workflow and when your analysis produces a result. In real-time recording systems, keeping this delay short is critical.
41. **Keeping Stages Independent:** Designing a workflow so that the data analysis stage does not depend on the visualization stage, and vice versa. This means you can update or replace one stage without breaking the others.
42. **Text Files vs. Optimized Data Files:** Saving data as plain text (such as CSV) is easy to read in a spreadsheet but very slow for large datasets. Optimized formats (such as HDF5 or NumPy binary files) are much faster to load and take up less disk space.
43. **Simultaneous Processing (Parallel Computing):** Splitting a large task — such as analyzing 50 participants — into smaller chunks that run at the same time across multiple processor cores, rather than one after another.
44. **Hidden Configuration Settings:** Values that a workflow needs — such as access keys for a data repository — that are stored securely in the operating system rather than written directly into the code, to prevent accidental exposure.
45. **Software Libraries (Dependencies):** Pre-built collections of code (such as NumPy, SciPy, or scikit-learn) that provide ready-made tools for common tasks — so you do not have to write mathematical functions from scratch.
46. **Code Version Tracking (Git):** A system that records every change made to a set of code files over time, along with who made the change and when. This allows teams to collaborate and to restore earlier versions if something goes wrong.
47. **Error Handling:** Code that anticipates things going wrong — such as a missing file or a calculation that produces an undefined result — and responds gracefully rather than crashing the entire workflow.
48. **Data Array:** A structured list of numbers organized so that a computer can perform calculations on all of them efficiently — the basic building block of scientific data analysis.
49. **Data Backlog:** What happens when data arrives faster than your workflow can process it — causing a growing queue that eventually uses up available memory.
50. **Protecting Original Data:** A core rule of reproducible research: never modify your raw data files. Always write processed results to a new, separate file so the original record remains intact.


---

## ✅ Self-Check: What Did You Learn?

*Fill in the table below after completing both activities. Write the letter of the code cell (A.1, A.2, B.1, B.2) where your generated code best demonstrates each concept.*

| Concept Area | Concept Number | Where You Demonstrated It |
|---|---|---|
| Neuroscience | Concept 1 — What a BMI does | [ Enter Cell ID ] |
| Neuroscience | Concept 3 — Why signal delay matters | [ Enter Cell ID ] |
| Neuroscience | Concept 5 — How fMRI works | [ Enter Cell ID ] |
| Computing | Concept 26 — Working memory vs. storage | [ Enter Cell ID ] |
| Computing | Concept 36 — Organizing code into stages | [ Enter Cell ID ] |
| Computing | Concept 41 — Keeping stages independent | [ Enter Cell ID ] |
